# MCBench: Measuring Metacognitive Capability in AI via Signal Detection Theory

**Hackathon:** Measuring Progress Toward AGI – Cognitive Abilities  
**Author:** Ivo Georgiev  
**Dataset:** MMLU-Pro + GPQA Diamond (600 eval items)

## What MCBench measures

MCBench measures **metacognitive capability** — not just whether a model gets answers right,
but whether it *knows what it knows*. This is a key AGI milestone: human-level metacognition
allows us to allocate cognitive effort, recognise uncertainty, and avoid confident mistakes.

### Two-turn evaluation protocol (solo models)
| Turn | Task | What it measures |
|------|------|------------------|
| 1 | Zero-shot MCQ answer | Type 1 accuracy → d′ |
| 2 | Confidence elicitation (0–100) | Metacognitive sensitivity → meta-d′ |

### MetaMind: multi-agent metacognition
MetaMind wraps any solo model in a three-agent pipeline (ToM → Domain → Response)
and tests whether multi-agent coordination improves metacognitive efficiency.
Run `mcbench_metamind` against the same model to get a directly comparable score.

### Metacognitive Capability Index (MCI)
$$\text{MCI} = 0.65 \cdot \max(0, M\text{-ratio}) + 0.35 \cdot (1 - \text{ECE})$$

where $M\text{-ratio} = \text{meta-}d' / d'$ (Fleming & Lau 2014 via Type 2 AUROC).

In [ ]:
# ── Setup ─────────────────────────────────────────────────────────────────────
import sys, re
from pathlib import Path

import kaggle_benchmarks as kbench
import pandas as pd
import numpy as np

# Add dataset root to path so src/ is importable on Kaggle
REPO_ROOT = Path('/kaggle/input/mcbench-metacognitive-benchmark')
if not REPO_ROOT.exists():
    REPO_ROOT = Path('.').resolve()   # local fallback
sys.path.insert(0, str(REPO_ROOT))

DATA_DIR = REPO_ROOT / 'data'
eval_df  = pd.read_parquet(DATA_DIR / 'eval_set.parquet')

print(f'Repo root : {REPO_ROOT}')
print(f'Eval set  : {len(eval_df)} items')
print(f'Domains   : {eval_df["domain"].value_counts().to_dict()}')

In [ ]:
# ── Helpers ───────────────────────────────────────────────────────────────────
def _extract_confidence(text: str) -> float:
    """Return the last 0-100 integer found in text, defaulting to 50."""
    for c in reversed(re.findall(r'\b(\d{1,3})\b', text)):
        val = int(c)
        if 0 <= val <= 100:
            return float(val)
    return 50.0


def _judge_accuracy(response: str, correct_answer: str) -> int:
    """Return 1 if the correct answer letter appears in the response."""
    return int(bool(
        re.search(rf'\b{re.escape(correct_answer.strip())}\b',
                  response, re.IGNORECASE)
    ))


def _run_two_turn(llm, df: pd.DataFrame) -> tuple[list[int], list[float]]:
    """
    Core two-turn evaluation loop shared by solo and MetaMind tasks.
    Returns (accuracy_list, confidence_list).
    """
    accuracy_list, confidence_list = [], []
    for _, row in df.iterrows():
        response1  = llm.prompt(row['main_prompt'])
        is_correct = _judge_accuracy(response1, row['correct_answer'])
        turn2 = (
            row['main_prompt']
            + f"\n\nYour previous answer was:\n{response1}\n\n"
            + row['meta_question']
        )
        response2  = llm.prompt(turn2)
        confidence = _extract_confidence(response2)
        accuracy_list.append(is_correct)
        confidence_list.append(confidence)
    return accuracy_list, confidence_list


def _compute_and_save(accuracy_list, confidence_list, df, tag: str) -> float:
    """Compute MCI, print summary, save detail CSV, return MCI float."""
    from src.metrics.mci import compute_mci
    from src.metrics.sdt import compute_m_ratio

    # SRS (self-recognition) requires multi-model decoys — use random baseline.
    mci_result = compute_mci(accuracy_list, confidence_list, srs=1/3)
    sdt_result = compute_m_ratio(accuracy_list, confidence_list)

    detail = df[['id', 'domain', 'source']].copy()
    detail['accuracy']   = accuracy_list
    detail['confidence'] = confidence_list
    detail.to_csv(f'mcbench_detail_{tag}.csv', index=False)

    print(f"[{tag}] MCI={mci_result['mci']:.4f}  d'={sdt_result['d_prime']:.4f}"
          f"  M-ratio={sdt_result['m_ratio']:.4f}  ECE={mci_result['ece']:.4f}"
          f"  acc={sum(accuracy_list)/len(accuracy_list):.3f}")
    return float(mci_result['mci'])

In [ ]:
# ── Task 1: solo model metacognition ─────────────────────────────────────────
@kbench.task(name="mcbench_metacognition")
def mcbench_metacognition(llm, df: pd.DataFrame) -> float:
    """
    Standard two-turn MCBench evaluation for a single model.
    Turn 1: zero-shot MCQ answer.  Turn 2: confidence 0-100.
    Returns MCI (higher = stronger metacognitive capability).
    """
    with kbench.client.enable_cache():
        acc, conf = _run_two_turn(llm, df)
    return _compute_and_save(acc, conf, df, tag='solo')

In [ ]:
# ── Task 2: MetaMind three-agent metacognition ────────────────────────────────
#
# MetaMind uses the same underlying model (kbench.llm) but routes each question
# through a three-agent pipeline before answering:
#   Agent 1 (ToM)    — generate N perspectives on who might answer correctly
#   Agent 2 (Domain) — synthesise perspectives into a consensus answer
#   Agent 3 (Response) — produce the final answer from the consensus
# Turn 2 confidence uses the Domain agent's agreement score as a prior.
#
# Run this task against the same model as mcbench_metacognition to measure
# the multi-agent metacognitive advantage (ΔMCI = MetaMind - solo).

_TOM_TEMPLATE = """You are a Theory-of-Mind reasoning agent.
For the question below, generate {n} brief perspectives describing the type of
reasoner most likely to answer correctly, and what approach they would take.
Be concise — one sentence per perspective.

Question: {question}

Perspectives:"""

_DOMAIN_TEMPLATE = """You are a domain expert evaluating multiple reasoning perspectives.
Given the question and the perspectives below, determine the most likely correct answer.
End your response with exactly these two lines:
CONSENSUS_ANSWER: <single letter A/B/C/D>
AGREEMENT_SCORE: <integer 0-100>

Question: {question}

Perspectives:
{perspectives}

Analysis:"""

_RESPONSE_TEMPLATE = """Answer the following question. A panel of experts reached consensus
that the answer is most likely {consensus} (agreement: {agreement}/100).
Use this as a prior but reason independently.

{question}"""

_CONFIDENCE_TEMPLATE = """You previously answered:
{question}

Your answer: {answer}

A reasoning panel had agreement score {agreement}/100 on this answer.

{meta_question}"""


def _metamind_two_turn(
    llm,
    question: str,
    correct_answer: str,
    meta_question: str,
    n_hypotheses: int = 3,
) -> tuple[int, float]:
    """One question through the MetaMind pipeline. Returns (is_correct, confidence)."""
    # Agent 1: Theory-of-Mind perspectives
    perspectives = llm.prompt(
        _TOM_TEMPLATE.format(n=n_hypotheses, question=question)
    )

    # Agent 2: Domain consensus
    domain_out = llm.prompt(
        _DOMAIN_TEMPLATE.format(question=question, perspectives=perspectives)
    )
    # Parse CONSENSUS_ANSWER and AGREEMENT_SCORE from domain output
    consensus_match  = re.search(r'CONSENSUS_ANSWER:\s*([A-Da-d])', domain_out)
    agreement_match  = re.search(r'AGREEMENT_SCORE:\s*(\d+)',       domain_out)
    consensus  = consensus_match.group(1).upper() if consensus_match else 'A'
    agreement  = int(agreement_match.group(1))    if agreement_match else 50
    agreement  = max(0, min(100, agreement))

    # Agent 3: Final answer informed by consensus
    response = llm.prompt(
        _RESPONSE_TEMPLATE.format(
            question=question, consensus=consensus, agreement=agreement
        )
    )
    is_correct = _judge_accuracy(response, correct_answer)

    # Turn 2: confidence elicitation with agreement prior
    conf_response = llm.prompt(
        _CONFIDENCE_TEMPLATE.format(
            question=question, answer=response,
            agreement=agreement, meta_question=meta_question
        )
    )
    confidence = _extract_confidence(conf_response)
    return is_correct, confidence


@kbench.task(name="mcbench_metamind")
def mcbench_metamind(llm, df: pd.DataFrame) -> float:
    """
    MetaMind three-agent MCBench evaluation.
    Uses the same model as mcbench_metacognition but routes each question
    through a ToM → Domain → Response pipeline.
    Compare ΔMCI = mcbench_metamind - mcbench_metacognition on the same model.
    """
    accuracy_list, confidence_list = [], []
    with kbench.client.enable_cache():
        for _, row in df.iterrows():
            is_correct, confidence = _metamind_two_turn(
                llm,
                question=row['main_prompt'],
                correct_answer=row['correct_answer'],
                meta_question=row['meta_question'],
            )
            accuracy_list.append(is_correct)
            confidence_list.append(confidence)
    return _compute_and_save(accuracy_list, confidence_list, df, tag='metamind')

In [ ]:
# ── Select solo task as the primary benchmark task ────────────────────────────
# MetaMind can be run separately (cell below) and submitted as a second entry.
%choose mcbench_metacognition

In [ ]:
# ── Run solo benchmark ────────────────────────────────────────────────────────
# kbench.llm = model selected via Kaggle model picker.
# For a smoke test use eval_df.head(50).
solo_mci = mcbench_metacognition.run(kbench.llm, eval_df)
print(f'Solo MCI: {solo_mci:.4f}')

In [ ]:
# ── (Optional) Run MetaMind on the same model ─────────────────────────────────
# Uncomment to measure the multi-agent metacognitive advantage.
# Uses ~4× as many LLM calls as the solo task.
#
# metamind_mci = mcbench_metamind.run(kbench.llm, eval_df)
# print(f'MetaMind MCI : {metamind_mci:.4f}')
# print(f'ΔMCI (MetaMind - solo) : {metamind_mci - solo_mci:+.4f}')

In [ ]:
# ── Results summary ───────────────────────────────────────────────────────────
from src.metrics.sdt import compute_m_ratio

detail = pd.read_csv('mcbench_detail_solo.csv')
acc    = detail['accuracy'].tolist()
conf   = detail['confidence'].tolist()
sdt    = compute_m_ratio(acc, conf)

summary = pd.DataFrame([{
    'model':        str(kbench.llm),
    'mci':          solo_mci,
    'd_prime':      sdt['d_prime'],
    'm_ratio':      sdt['m_ratio'],
    'type2_auroc':  sdt['type2_auroc'],
    'accuracy':     sum(acc) / len(acc),
    'mean_conf':    sum(conf) / len(conf),
}])

display(summary.style
    .format({c: '{:.4f}' for c in ['mci','d_prime','m_ratio','type2_auroc','accuracy','mean_conf']})
    .set_caption('MCBench Result')
)

summary.to_csv('mcbench_leaderboard.csv', index=False)
print('Saved: mcbench_leaderboard.csv  mcbench_detail_solo.csv')

In [ ]:
# ── Calibration plots ─────────────────────────────────────────────────────────
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle('MCBench: Metacognitive Capability Profile', fontsize=12)

correct_conf   = [c for c, a in zip(conf, acc) if a == 1]
incorrect_conf = [c for c, a in zip(conf, acc) if a == 0]
axes[0].hist(correct_conf,   bins=20, alpha=0.6, color='steelblue', label='Correct')
axes[0].hist(incorrect_conf, bins=20, alpha=0.6, color='tomato',    label='Incorrect')
axes[0].set_xlabel('Stated Confidence (0–100)')
axes[0].set_ylabel('Count')
axes[0].set_title('Confidence by Answer Correctness')
axes[0].legend()
axes[0].axvline(50, color='grey', linestyle='--', alpha=0.5)

bins_edges  = np.linspace(0, 100, 11)
conf_arr    = np.array(conf)
acc_arr     = np.array(acc, dtype=float)
bin_centers, bin_acc = [], []
for lo, hi in zip(bins_edges[:-1], bins_edges[1:]):
    mask = (conf_arr >= lo) & (conf_arr < hi)
    if mask.sum() > 0:
        bin_centers.append((lo + hi) / 2)
        bin_acc.append(acc_arr[mask].mean() * 100)
axes[1].plot(bin_centers, bin_acc, 'o-', color='darkorange', label='Model')
axes[1].plot([0, 100], [0, 100], 'k--', alpha=0.4, label='Perfect calibration')
axes[1].set_xlabel('Stated Confidence (0–100)')
axes[1].set_ylabel('Actual Accuracy (%)')
axes[1].set_title(f"Calibration Curve  (M-ratio={sdt['m_ratio']:.3f})")
axes[1].legend()

plt.tight_layout()
plt.savefig('mcbench_calibration.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: mcbench_calibration.png')